# LatentSaccade EmuVLA — Colab Evaluation

**Method 6**: 임베딩 레벨 공간 어텐션 + Saccade

- 픽셀/VQ 토큰 수정 없음 → OOD 제로
- `embed_tokens` forward hook으로 **현재 프레임 시각 토큰만** 가중치 조절
- DINO bbox → fovea(1.0) / secondary(0.5) / 배경(0.2)
- Saccade: GRASP→PLACE 전환 시 fovea 대상 전환

---
**실행 순서**: Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5

In [ ]:
# ── Cell 1: 경로 설정 (여기만 수정하면 됨) ─────────────────────────────────────

# UniVLA 모델 가중치
EMU_HUB    = "/content/pretrain/UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K"
VISION_HUB = "/content/pretrain/Emu3-VisionTokenizer"
FAST_PATH  = "/content/UniVLA/pretrain/fast_bridge_t5_s50"  # tokenizer 디렉토리 직접 지정

# 평가 설정
TASK          = "widowx_put_eggplant_in_basket"   # SimplerEnv 태스크 이름
N_EPISODES    = 10                                 # 평가할 에피소드 수
SAVE_VIDEO    = True                               # GIF 저장 여부
OUTPUT_DIR    = "/content/latent_saccade_eval"
DINO_DEBUG    = None   # "/content/dino_debug"  ← DINO 검출 시각화 저장할 경우

# LatentSaccade 하이퍼파라미터
BG_WEIGHT         = 0.2   # 배경 토큰 임베딩 스케일 (0~1)
PLACE_SRC_WEIGHT  = 0.5   # PLACE 페이즈 소스 객체 스케일
MIN_GRASP_STEPS   = 15    # GRASP 페이즈 최소 스텝 (saccade 발동 전)
CONSEC_CLOSE      = 3     # 그리퍼 닫힘 연속 판정 횟수
DINO_CACHE_STEPS  = 5     # DINO 재검출 주기 (스텝)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 경로 검증
for p, name in [(EMU_HUB, "EMU_HUB"), (VISION_HUB, "VISION_HUB"), (FAST_PATH, "FAST_PATH")]:
    status = "OK" if os.path.exists(p) else "NOT FOUND"
    print(f"[{status}] {name}: {p}")

In [ ]:
# ── Cell 2: 환경 설치 (세션당 1회) ────────────────────────────────────────────
# 의존성: peft, numpy<2 (cv2/NumPy 2.x 충돌 방지)

!conda run -n univla pip install -q peft "numpy<2"

# 최신 코드 pull
!git -C /content/UniVLA pull origin claude/foveated-tokenization-experiment-EQyFt

print("[OK] 설치 완료")

In [ ]:
# ── Cell 3: SimplerEnv 설치 (세션당 1회) ─────────────────────────────────────
# SimplerEnv가 이미 /content/SimplerEnv에 있으면 스킵

import os
if not os.path.exists("/content/SimplerEnv"):
    !git clone --quiet https://github.com/simpler-env/SimplerEnv /content/SimplerEnv
    !pip install -q -e /content/SimplerEnv
    !pip install -q -e /content/SimplerEnv/ManiSkill2_real2sim
    print("[OK] SimplerEnv 설치 완료")
else:
    print("[OK] SimplerEnv 이미 설치됨")

In [ ]:
# ── Cell 4: 모델 로드 ─────────────────────────────────────────────────────────
# LatentSaccadeEmuVLAInference 인스턴스 생성
# 소요 시간: ~2-3분 (Emu3MoE 로드)

import sys, os

# 경로 등록
_ROOT     = "/content/UniVLA"
_EXP      = os.path.join(_ROOT, "experiments", "foveated_tokenization")
_EMU3     = os.path.join(_ROOT, "reference", "Emu3")
_SIMPLER  = "/content/SimplerEnv"
_MANSKILL = os.path.join(_SIMPLER, "ManiSkill2_real2sim")

for p in [_ROOT, _EXP, _EMU3, _SIMPLER, _MANSKILL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# lightning stub (calvin model_wrapper 임포트 시 필요)
import types
if "lightning" not in sys.modules:
    for _n in ["lightning", "lightning.pytorch", "lightning.pytorch.trainer"]:
        sys.modules.setdefault(_n, types.ModuleType(_n))
    class _Trainer: pass
    sys.modules["lightning.pytorch.trainer"].Trainer = _Trainer

import torch
from foveated_inference import LatentSaccadeEmuVLAInference

model = LatentSaccadeEmuVLAInference(
    emu_hub              = EMU_HUB,
    vq_hub               = VISION_HUB,
    vision_hub           = VISION_HUB,
    device               = "cuda",
    policy_setup         = "widowx_bridge",
    fast_path            = FAST_PATH,
    dino_model           = "IDEA-Research/grounding-dino-tiny",
    dino_cache_steps     = DINO_CACHE_STEPS,
    box_threshold        = 0.15,
    text_threshold       = 0.15,
    bbox_margin          = 2,
    bg_weight            = BG_WEIGHT,
    place_src_weight     = PLACE_SRC_WEIGHT,
    min_grasp_steps      = MIN_GRASP_STEPS,
    consecutive_close_required = CONSEC_CLOSE,
    enable_latent_mask   = True,
    dino_debug_dir       = DINO_DEBUG,
)

print(f"[OK] LatentSaccadeEmuVLAInference 로드 완료")
print(f"     vis_start={model.vis_start}  vis_end={model.vis_end}")
print(f"     device={next(model.model.parameters()).device}")

In [ ]:
# ── Cell 5: SimplerEnv 평가 실행 ─────────────────────────────────────────────
# run_eval_compare.py의 evaluate_model / run_single_episode 재사용
# 소요 시간: 에피소드당 ~2-4분 × N_EPISODES

import json, time
import numpy as np
from PIL import Image as _PIL

# ── task config ────────────────────────────────────────────────────────────
TASK_CONFIGS = {
    "widowx_put_eggplant_in_basket": {
        "env_name": "PutEggplantInBasketScene-v0",
        "robot": "widowx_sink_camera_setup",
        "scene_name": "bridge_table_1_v2",
        "rgb_overlay_path": "ManiSkill2_real2sim/data/real_inpainting/bridge_sink.png",
        "rgb_overlay_cameras": ["3rd_view_camera"],
        "obj_variation_mode": "episode",
        "obj_episode_range": [0, 24],
        "obs_camera_name": "3rd_view_camera",
        "control_freq": 3, "sim_freq": 513, "max_episode_steps": 80,
    },
    "widowx_carrot_on_plate": {
        "env_name": "PutCarrotOnPlateInScene-v0",
        "robot": "widowx_sink_camera_setup",
        "scene_name": "bridge_table_1_v2",
        "rgb_overlay_path": "ManiSkill2_real2sim/data/real_inpainting/bridge_sink.png",
        "rgb_overlay_cameras": ["3rd_view_camera"],
        "obj_variation_mode": "episode",
        "obj_episode_range": [0, 24],
        "obs_camera_name": "3rd_view_camera",
        "control_freq": 3, "sim_freq": 513, "max_episode_steps": 80,
    },
    "widowx_stack_cube": {
        "env_name": "StackGreenCubeOnYellowCubeBakedTexInScene-v0",
        "robot": "widowx_sink_camera_setup",
        "scene_name": "bridge_table_1_v1",
        "rgb_overlay_path": "ManiSkill2_real2sim/data/real_inpainting/bridge_real_eval_1.png",
        "rgb_overlay_cameras": ["3rd_view_camera"],
        "obj_variation_mode": "episode",
        "obj_episode_range": [0, 24],
        "obs_camera_name": "3rd_view_camera",
        "control_freq": 3, "sim_freq": 513, "max_episode_steps": 60,
    },
}

task_cfg = TASK_CONFIGS[TASK]
print(f"[eval] task={TASK}  n_episodes={N_EPISODES}")

# ── env helpers ────────────────────────────────────────────────────────────
def _build_env(task_cfg, ep_id):
    from simpler_env.utils.env.env_builder import build_maniskill2_env, get_robot_control_mode
    robot = task_cfg["robot"]
    control_mode = get_robot_control_mode(robot, "emu_vla")
    kw = dict(
        obs_mode="rgbd", robot=robot,
        sim_freq=task_cfg["sim_freq"], control_mode=control_mode,
        control_freq=task_cfg["control_freq"],
        max_episode_steps=task_cfg["max_episode_steps"],
    )
    if task_cfg.get("scene_name"):
        kw["scene_name"] = task_cfg["scene_name"]
    overlay = task_cfg.get("rgb_overlay_path")
    if overlay:
        for base in ["/content/SimplerEnv", "/content/SimplerEnv/ManiSkill2_real2sim/.."]:
            candidate = os.path.join(base, overlay)
            if os.path.exists(candidate):
                kw["rgb_overlay_path"] = candidate
                kw["rgb_overlay_cameras"] = task_cfg.get("rgb_overlay_cameras", ["3rd_view_camera"])
                kw["camera_cfgs"] = {"add_segmentation": True}
                break
    env = build_maniskill2_env(task_cfg["env_name"], **kw)
    obs, _ = env.reset(options={"obj_init_options": {"episode_id": ep_id}})
    return env, obs

def _get_image(env, obs, cam_name):
    from simpler_env.utils.env.observation_utils import get_image_from_maniskill2_obs_dict
    return get_image_from_maniskill2_obs_dict(env, obs, camera_name=cam_name)

# ── episode loop ───────────────────────────────────────────────────────────
base_ids = list(range(*task_cfg["obj_episode_range"]))
ep_ids   = [base_ids[i % len(base_ids)] for i in range(N_EPISODES)]
cam_name = task_cfg["obs_camera_name"]

results = []
for ep_count, ep_id in enumerate(ep_ids):
    print(f"\n{'='*50}")
    print(f"  Episode {ep_count} (env_id={ep_id})")
    print(f"{'='*50}")

    env, obs = _build_env(task_cfg, ep_id)
    instruction = env.get_language_instruction()
    image = _get_image(env, obs, cam_name)
    print(f"  instruction: {instruction}")

    model.reset()
    frames = [image.copy()] if SAVE_VIDEO else []
    done = truncated = False
    step = 0
    t0 = time.time()

    while not (done or truncated) and step < task_cfg["max_episode_steps"]:
        raw_actions, env_actions = model.step(image, instruction)
        for raw_a, env_a in zip(raw_actions, env_actions):
            obs, _, done, truncated, info = env.step(
                np.concatenate([
                    env_a["world_vector"],
                    env_a["rot_axangle"],
                    env_a["gripper"],
                ])
            )
            image = _get_image(env, obs, cam_name)
            if SAVE_VIDEO and step % 4 == 0:
                frames.append(image.copy())

            new_instr = env.get_language_instruction()
            if new_instr != instruction:
                instruction = new_instr
                model.reset()

            step += 1
            if done or truncated:
                break

    elapsed = time.time() - t0
    status  = "SUCCESS" if done else "FAIL"
    print(f"  → {status}  ({step} steps, {elapsed:.1f}s)")
    env.close()

    if SAVE_VIDEO and frames:
        vpath = os.path.join(OUTPUT_DIR, f"ep{ep_count:02d}_{status.lower()}.gif")
        pils = [_PIL.fromarray(f) for f in frames]
        pils[0].save(vpath, save_all=True, append_images=pils[1:], loop=0, duration=100)
        print(f"  GIF 저장: {vpath}")

    results.append({"ep_count": ep_count, "ep_id": ep_id,
                    "success": bool(done), "steps": step, "elapsed": elapsed})

# ── 최종 결과 ──────────────────────────────────────────────────────────────
n_success  = sum(r["success"] for r in results)
success_rate = n_success / len(results)
avg_steps    = np.mean([r["steps"]   for r in results])
avg_time     = np.mean([r["elapsed"] for r in results])

print(f"\n{'='*50}")
print(f"  LatentSaccade 평가 완료")
print(f"  성공률: {n_success}/{len(results)} = {success_rate:.1%}")
print(f"  평균 스텝: {avg_steps:.0f}  평균 시간: {avg_time:.1f}s/ep")
print(f"{'='*50}")
for r in results:
    s = "✓" if r["success"] else "✗"
    print(f"  {s} ep{r['ep_count']:02d} (id={r['ep_id']}): {r['steps']} steps")

# 결과 저장
summary = {
    "model": "LatentSaccadeEmuVLAInference",
    "task": TASK,
    "n_episodes": len(results),
    "success_rate": success_rate,
    "avg_steps": float(avg_steps),
    "avg_time": float(avg_time),
    "config": {
        "bg_weight": BG_WEIGHT,
        "place_src_weight": PLACE_SRC_WEIGHT,
        "min_grasp_steps": MIN_GRASP_STEPS,
        "consecutive_close": CONSEC_CLOSE,
        "dino_cache_steps": DINO_CACHE_STEPS,
    },
    "episodes": results,
}
save_path = os.path.join(OUTPUT_DIR, f"results_{TASK}.json")
with open(save_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n결과 저장: {save_path}")

In [ ]:
# ── Cell 6 (선택): run_eval_compare.py 스크립트로 실행 ────────────────────────
# Cell 5 대신 스크립트 방식을 원할 때 사용
# --latent-saccade-only 플래그로 해당 모델만 실행

!conda run -n univla python \
    /content/UniVLA/experiments/foveated_tokenization/run_eval_compare.py \
    --emu-hub      "{EMU_HUB}" \
    --vq-hub       "{VISION_HUB}" \
    --vision-hub   "{VISION_HUB}" \
    --fast-path    "{FAST_PATH}" \
    --task         "{TASK}" \
    --n-episodes   {N_EPISODES} \
    --output-dir   "{OUTPUT_DIR}" \
    --latent-saccade-only \
    --bg-weight    {BG_WEIGHT} \
    --place-src-weight {PLACE_SRC_WEIGHT} \
    --min-grasp-steps  {MIN_GRASP_STEPS} \
    --consecutive-close {CONSEC_CLOSE} \
    --dino-cache-steps  {DINO_CACHE_STEPS} \
    --save-video

In [ ]:
# ── Cell 7 (선택): ablation — latent mask 비활성화 (saccade만 적용) ─────────────
# enable_latent_mask=False → 임베딩 가중치 없이 DINO+saccade만 동작
# token_saccade 대비 순수 saccade 효과 분리 가능

from foveated_inference import LatentSaccadeEmuVLAInference
import gc, torch

ablation_model = LatentSaccadeEmuVLAInference(
    emu_hub              = EMU_HUB,
    vq_hub               = VISION_HUB,
    vision_hub           = VISION_HUB,
    device               = "cuda",
    policy_setup         = "widowx_bridge",
    fast_path            = FAST_PATH,
    min_grasp_steps      = MIN_GRASP_STEPS,
    consecutive_close_required = CONSEC_CLOSE,
    enable_latent_mask   = False,   # ← 임베딩 마스크 OFF
)

# 위 Cell 5의 루프를 ablation_model로 동일하게 실행하면
# "saccade 효과" vs "saccade + embedding mask 효과" 를 비교할 수 있음
print("[OK] ablation model 로드 완료 (enable_latent_mask=False)")